# On-the-fly curriculum runs (nanoTabPFN) — Colab T4

Runs `onthefly_baseline_full` vs `onthefly_curriculum_features` (3 seeds each) and prints the all/binary/multiclass ROC-AUC summary.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run the cells top to bottom. Results are saved to Google Drive after each run, and the notebook **skips anything already done**, so a disconnect never wastes work — just reopen and re-run.

## Cell 1 — setup (clone repos at pinned commits + install, ~3–5 min)

In [ ]:
%cd /content
!git clone -b second-wave https://github.com/parsafrei-droid/ml-lab-curriculum.git
%cd /content/ml-lab-curriculum
!git clone https://github.com/automl/TFM-Playground.git && (cd TFM-Playground && git checkout 98e33be)
!git clone https://github.com/soda-inria/tabicl.git      && (cd tabicl && git checkout 8f665ed)
!pip -q install -e "./tabicl[pretrain]" pfns==0.3.0 h5py schedulefree==1.4.1 openml==0.15.1 pyyaml scipy scikit-learn
# --- patch: an import fix that exists only in the cluster's working tree, never committed to TFM-Playground ---
import pathlib
_f = pathlib.Path("/content/ml-lab-curriculum/TFM-Playground/tfmplayground/external_priors/tabicl.py")
_f.write_text(_f.read_text().replace("from tabicl.prior.dataset import PriorDataset", "from tabicl.prior import PriorDataset"))
print("import patch applied:", "from tabicl.prior import PriorDataset" in _f.read_text())
# NOTE: torch/numpy left at Colab's preinstalled versions (we do NOT force torch==2.9 — avoids breaking CUDA).

## Cell 2 — GPU check + mount Drive (so finished runs survive a disconnect)

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — set Runtime>Change runtime type>T4 GPU")
from google.colab import drive; drive.mount('/content/drive')
import os; SAVE="/content/drive/MyDrive/curriculum_results"; os.makedirs(SAVE, exist_ok=True)
print("saving results to", SAVE)

## Cell 3 — resumable runner (skips runs already saved to Drive)

In [ ]:
%cd /content/ml-lab-curriculum/Second_Wave
import time, subprocess, shutil, os, glob
SAVE="/content/drive/MyDrive/curriculum_results"
# restore results finished in a previous session so plot.py / summary see them all
for d in glob.glob(f"{SAVE}/onthefly_*"):
    name=os.path.basename(d); os.makedirs(f"results/{name}", exist_ok=True)
    for f in glob.glob(f"{d}/*"): shutil.copy(f, f"results/{name}/")
def run(cfg, seed):
    name=f"{cfg}_s{seed}"
    if os.path.exists(f"{SAVE}/{name}/tabarena_scores.json"):
        print(f"==== {name} already done (found in Drive) — skipping ===="); return
    t=time.time()
    subprocess.run(["python","train.py","--config",f"configs/{cfg}.yaml","--name",name,"--seed",str(seed)], check=True)
    subprocess.run(["python","evaluate.py","--checkpoint",f"results/{name}/checkpoint.pth","--max-n-samples","5000"], check=True)
    d=f"{SAVE}/{name}"; os.makedirs(d, exist_ok=True)
    for f in ("log.csv","tabarena_scores.json","meta.json","config.yaml"):
        p=f"results/{name}/{f}"
        if os.path.exists(p): shutil.copy(p, d)
    print(f"==== {name} done in {(time.time()-t)/60:.1f} min ====")

## Cell 4 — SMOKE TEST: one pair first, to measure real T4 speed
Read the "done in X min". Multiply by ~4 to estimate the remaining 4 runs.

In [ ]:
run("onthefly_baseline_full", 42)
run("onthefly_curriculum_features", 42)

## Cell 5 — the remaining 4 runs (seeds 1 and 2)
Safe to re-run after a disconnect — finished seeds are skipped.

In [ ]:
for seed in (1, 2):
    run("onthefly_baseline_full", seed)
    run("onthefly_curriculum_features", seed)

## Cell 6 — figures + summary table (this is what you submit)

In [ ]:
import subprocess, shutil, json, glob, statistics as st
subprocess.run(["python","plot.py"], check=True)
shutil.copytree("figures", f"{SAVE}/figures", dirs_exist_ok=True)
rows={}
for f in sorted(glob.glob("results/onthefly_*/tabarena_scores.json")):
    n=f.split('/')[1]; cfg=n.rsplit('_s',1)[0]
    rows.setdefault(cfg, []).append(json.load(open(f))["summary"])
print(f"{'config':32s} {'scope':11s} mean_roc_auc  n_seeds")
for cfg, lst in rows.items():
    for scope in ("all","binary","multiclass"):
        vals=[x[scope]["roc_auc"] for x in lst if x[scope]["roc_auc"] is not None]
        if vals: print(f"{cfg:32s} {scope:11s} {st.mean(vals):.4f}        {len(vals)}")